# 🤖 Xiangqi-R1: Multi-Million Self-Play Dataset Mining & Qwen 2.5/3.5 GRPO Training (Google Colab)
### Khai Thác Dữ Liệu Tự Đấu Quy Mô Lớn & Huấn Luyện Mô Hình AI Cờ Tướng Thế Hệ Mới Xiangqi-R1 (Qwen 0.5B / 0.8B / 7B) bằng GRPO

- **HuggingFace Dataset Repo**: [hoduyquocbao/xiangqi-r1-dataset](https://huggingface.co/datasets/hoduyquocbao/xiangqi-r1-dataset)
- **HuggingFace Model 0.5B**: [hoduyquocbao/xiangqi-r1-0.5b](https://huggingface.co/hoduyquocbao/xiangqi-r1-0.5b)
- **HuggingFace Model 0.8B**: [hoduyquocbao/xiangqi-r1-0.8b](https://huggingface.co/hoduyquocbao/xiangqi-r1-0.8b)
- **HuggingFace Model 7B**: [hoduyquocbao/xiangqi-r1](https://huggingface.co/hoduyquocbao/xiangqi-r1)


In [ ]:
# 1. Kiểm tra phần cứng GPU & Khai báo môi trường
import os, sys, time, torch

HAS_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if HAS_CUDA else "cpu")

if HAS_CUDA:
    torch.cuda.synchronize()
    torch.cuda.empty_cache()
    total_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"⚡ [GPU ACTIVE] Tesla T4 GPU khả dụng với {total_mem:.2f}GB VRAM thực tế!")
else:
    print("⚠️ Không tìm thấy GPU CUDA. Đang chạy trên CPU Multi-Core.")

# Cài đặt Thư viện GPU Unsloth + TRL + HuggingFace
!nvidia-smi
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes datasets huggingface_hub

In [ ]:
# 2. Khai báo Token HuggingFace và Đăng nhập Hub
import os, sys, re, json, glob, torch
from huggingface_hub import login, HfApi
from unsloth import FastLanguageModel
from datasets import Dataset, load_dataset
from trl import GRPOTrainer, GRPOConfig

HF_TOKEN = os.environ.get("HF_TOKEN", "")
if HF_TOKEN:
    try:
        login(token=HF_TOKEN)
        print("✅ Đã đăng nhập HuggingFace Hub thành công!")
    except Exception as err:
        print(f"⚠️ Đăng nhập HuggingFace Hub thất bại: {err}")
else:
    print("⚠️ Không tìm thấy biến môi trường HF_TOKEN. Mô hình sẽ lưu cục bộ.")

In [ ]:
# 3. Chọn mô hình Qwen (0.5B, 0.8B, hoặc 7B)
VARIANT = os.environ.get("MODEL_VARIANT", "0.5b").lower()

if VARIANT == "7b":
    BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
    MODEL_REPO = "hoduyquocbao/xiangqi-r1"
elif VARIANT == "0.8b":
    BASE_MODEL = "hoduyquocbao/xiangqi-r1-0.8b"
    MODEL_REPO = "hoduyquocbao/xiangqi-r1-0.8b"
else:
    BASE_MODEL = "Qwen/Qwen2.5-Coder-0.5B-Instruct"
    MODEL_REPO = "hoduyquocbao/xiangqi-r1-0.5b"

DATASET_REPO = "hoduyquocbao/xiangqi-r1-dataset"

print(f"🚀 Base Model được chọn: {BASE_MODEL}")
print(f"📦 Dataset Target: https://huggingface.co/datasets/{DATASET_REPO}")
print(f"🤖 Model Target: https://huggingface.co/{MODEL_REPO}")

In [ ]:
# 4. Tải Mô hình Gốc Qwen & Cấu hình Unsloth FP16 Tensor Cores
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=1024,
    load_in_4bit=True,
    fast_inference=False,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
print(f"✅ Khởi tạo Unsloth 4-bit LoRA cho {BASE_MODEL} thành công!")

In [ ]:
# 5. Định nghĩa 3 Máy chấm điểm tự động (GRPO Reward Functions: syntax, rule, quality)

FORMAT = re.compile(r"^\s*<(thought|think)>.*?<\/(\1)>\s*\n?\s*([a-i][0-9][a-i][0-9])\s*$", re.DOTALL)
MOVE = re.compile(r"([a-i][0-9][a-i][0-9])")
FEN = re.compile(r"2\. Chuỗi Chuẩn FEN.*?:?\n([a-zA-Z0-9/]+\s+[wb]\s+-\s+-\s+\d+\s+\d+)")

def parse(fen):
    """Giải mã FEN thành ma trận 2D và bên đến lượt đi ('w' hoặc 'b')."""
    if not isinstance(fen, str):
        return None, None
    parts = fen.split()
    if len(parts) < 2:
        return None, None
    board = parts[0]
    active = parts[1]
    rows = board.split('/')
    if len(rows) != 10:
        return None, None
    grid = []
    for row in rows:
        line = []
        for ch in row:
            if ch.isdigit():
                line.extend(['.'] * int(ch))
            else:
                line.append(ch)
        if len(line) != 9:
            return None, None
        grid.append(line)
    return grid, active

def valid(fen, move):
    """Kiểm tra tính hợp lệ về mặt luật cờ của nước đi dựa trên FEN hiện tại."""
    if not isinstance(move, str) or len(move) != 4:
        return False
    if not (move[1].isdigit() and move[3].isdigit()):
        return False
    grid, active = parse(fen)
    if not grid or not active:
        return False
    scol = ord(move[0]) - ord('a')
    srank = int(move[1])
    tcol = ord(move[2]) - ord('a')
    trank = int(move[3])
    if not (0 <= scol <= 8 and 0 <= tcol <= 8 and 0 <= srank <= 9 and 0 <= trank <= 9):
        return False
    if scol == tcol and srank == trank:
        return False
    srow = 9 - srank
    trow = 9 - trank
    piece = grid[srow][scol]
    if piece in ('.', ' '):
        return False
    if (active == 'w' and not piece.isupper()) or (active == 'b' and not piece.islower()):
        return False
    target = grid[trow][tcol]
    if target not in ('.', ' '):
        if (piece.isupper() and target.isupper()) or (piece.islower() and target.islower()):
            return False
    kind = piece.upper()
    if kind in ('K', 'A'):
        if not (3 <= tcol <= 5):
            return False
        if piece.isupper() and not (0 <= trank <= 2):
            return False
        if piece.islower() and not (7 <= trank <= 9):
            return False
    elif kind == 'B':
        if piece.isupper() and trank > 4:
            return False
        if piece.islower() and trank < 5:
            return False
    elif kind == 'P':
        if piece.isupper():
            if trank < srank:
                return False
            if srank < 5 and (tcol != scol or trank <= srank):
                return False
        else:
            if trank > srank:
                return False
            if srank > 4 and (tcol != scol or trank >= srank):
                return False
    return True

def syntax(prompts, completions, **kwargs):
    rewards = []
    for completion in completions:
        text = completion.strip()
        if FORMAT.match(text):
            rewards.append(1.0)
        elif ("<thought>" in text and "</thought>" in text) or ("<think>" in text and "</think>" in text):
            if MOVE.search(text):
                rewards.append(0.5)
            else:
                rewards.append(0.0)
        else:
            rewards.append(-1.0)
    return rewards

def rule(prompts, completions, **kwargs):
    rewards = []
    for prompt, completion in zip(prompts, completions):
        text = completion.strip()
        match = MOVE.search(text)
        if not match:
            rewards.append(-0.5)
            continue
        move = match.group(1)
        matched = FEN.search(prompt)
        if matched:
            fen = matched.group(1)
            if valid(fen, move):
                rewards.append(2.0)
            else:
                rewards.append(-0.5)
        else:
            if len(move) == 4 and move[0] in "abcdefghi" and move[2] in "abcdefghi":
                rewards.append(1.0)
            else:
                rewards.append(-0.5)
    return rewards

def quality(prompts, completions, **kwargs):
    rewards = []
    grounds = kwargs.get("move", None)
    for idx, (prompt, completion) in enumerate(zip(prompts, completions)):
        text = completion.strip()
        match = MOVE.search(text)
        if not match:
            rewards.append(0.0)
            continue
        move = match.group(1)
        ground = grounds[idx] if grounds and idx < len(grounds) else None
        if ground and move == ground:
            rewards.append(3.0)
        elif move in ["b2e2", "h2e2", "b9c7", "h9g7", "c3c4", "g3g4"]:
            rewards.append(1.5)
        else:
            rewards.append(0.5)
    return rewards

print("✅ 3 GRPO Reward Functions ready (syntax, rule, quality)!")

In [ ]:
# 6. Kéo Dataset 3-in-1 đa chiều từ Hub (hoặc Fallback Cục Bộ)
try:
    print(f"📥 Đang tải dataset cờ tự đấu 3-in-1 từ HuggingFace Hub: {DATASET_REPO}...")
    dataset = load_dataset(DATASET_REPO, split="train")
    print(f"✅ Đã nạp thành công {len(dataset)} mẫu cờ tư duy sâu thực tế từ HuggingFace Hub!")
except Exception as err:
    print(f"⚠️ Không thể tải từ Hub ({err}), đang nạp dữ liệu cục bộ hợp nhất:")
    if os.path.exists("data/train.jsonl"):
        dataset = load_dataset("json", data_files="data/train.jsonl", split="train")
    else:
        files = sorted(glob.glob("data/real_mined_*.json"))
        if files:
            dataset = load_dataset("json", data_files=files, split="train")
        else:
            data = [{
                "prompt": "Trạng thái bàn cờ tướng... Đến lượt Đỏ đi...",
                "completion": "<thought>\n1. Phân tích FEN\n</thought>\nb2e2",
                "move": "b2e2",
                "stamp": 1700000000
            }] * 100
            dataset = Dataset.from_list(data)

In [ ]:
# 7. Cấu hình GRPOTrainer Tốc Độ Siêu Tốc (FP16 Tensor Cores)
args = GRPOConfig(
    output_dir=f"outputs/xiangqi-r1-{VARIANT}",
    learning_rate=1e-5,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_steps=5,
    lr_scheduler_type="cosine",
    optim="adamw_8bit",
    fp16=True,
    bf16=False,
    logging_steps=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    num_generations=4,
    max_prompt_length=512,
    max_completion_length=128,
    max_steps=200,
    save_steps=50,
    dataloader_num_workers=2,
    dataloader_pin_memory=True,
    report_to="none",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[syntax, rule, quality],
    args=args,
    train_dataset=dataset,
)

print("============================================================")
print(f"🚀 BẮT ĐẦU HUẤN LUYỆN GRPO XIANGQI-R1 ({VARIANT.upper()}) (FP16 TENSOR CORES)")
print("============================================================")
trainer.train()

In [ ]:
# 8. Xuất và Lưu Trọng số 16-bit Cục bộ & Đẩy lên HuggingFace Model Hub
output = f"outputs/xiangqi-r1-{VARIANT}-merged"
print(f"💾 Đang lưu trọng số hợp nhất 16-bit cục bộ tại: {output}...")
try:
    model.save_pretrained_merged(output, tokenizer, save_method="merged_16bit")
    print(f"✅ Đã lưu trọng số 16-bit cục bộ thành công!")
except Exception as err:
    print(f"⚠️ Không thể lưu mô hình cục bộ: {err}")

if HF_TOKEN:
    try:
        print(f"📤 Đang đẩy mô hình {VARIANT.upper()} lên HuggingFace Model Hub ({MODEL_REPO})...")
        model.push_to_hub_merged(MODEL_REPO, tokenizer, save_method="merged_16bit", token=HF_TOKEN)
        print(f"✅ HOÀN TẤT ĐĂNG TẢI XIANGQI-R1 LÊN HUB: https://huggingface.co/{MODEL_REPO}")
    except Exception as err:
        print(f"⚠️ Đăng tải Hub thất bại ({err}). Trọng số đã được lưu tại {output}.")
else:
    print(f"⚠️ Không có HF_TOKEN. Trọng số mô hình đã được bảo toàn tại: {output}")